
> **Violence Detect: Notifying from violence by WIFI**

**Steps:**
1. Mount Google Drive (for saving checkpoints)
2. Clone the repo
3. Install dependencies
4. Download dataset from Kaggle
5. Prepare dataset
6. Train
7. Evaluate

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Clone the project ───────────────────────────────────────────────
import os

# Option A: Clone from GitHub (if you push the refactored code)
# !git clone ...
# %cd WiFi-BullyDetect-refactored

# Option B: Upload zip to Drive and unzip
DRIVE_ROOT = '/content/drive/MyDrive/WiFi-BullyDetect'
PROJECT_DIR = '/content/WiFi-BullyDetect'

os.makedirs(PROJECT_DIR, exist_ok=True)

# If you uploaded the project zip to Drive:
# !unzip "/content/drive/MyDrive/WiFi-BullyDetect.zip" -d /content/

# Or copy individual files from Drive
# !cp -r "/content/drive/MyDrive/WiFi-BullyDetect/" /content/

%cd {PROJECT_DIR}
print("Working dir:", os.getcwd())

In [ ]:
# ── Step 3: Install dependencies ────────────────────────────────────────────
!pip install -q PyWavelets scikit-learn seaborn kaggle

In [ ]:
# ── Step 4: Setup Kaggle API ─────────────────────────────────────────────────
# Upload your kaggle.json here, OR place it in Drive and copy it
from google.colab import files

print("Upload your kaggle.json (from kaggle.com → Account → API Token)")
uploaded = files.upload()  # select kaggle.json

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured.')

In [ ]:
# ── Step 5: Download + prepare dataset ──────────────────────────────────────
!python scripts/prepare_dataset.py --kaggle --raw_dir data/raw --sliding_window

# Verify
!ls -la data/preprocessed/train/

In [ ]:
# ── Step 6: Check GPU ────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Step 7: (Optional) Adjust config ────────────────────────────────────────
# Edit config.py directly, or pass flags to train.py
# For Colab, set NUM_WORKERS=2 for faster data loading

config_patch = '''
# Patch config for Colab
import config
config.NUM_WORKERS = 2
config.PIN_MEMORY  = False
print("Config patched for Colab")
'''
exec(config_patch)

In [ ]:
# ── Step 8: Train ────────────────────────────────────────────────────────────
# Option A: run train.py as a script
!python train.py --run_name colab_dwt --model DWT --epochs 100 --batch 32 --workers 2

# Option B: run inline (useful to inspect outputs)
# import train as t
# import sys; sys.argv = ['train.py', '--run_name', 'colab_dwt']
# t.main()

In [ ]:
# ── Step 9: Evaluate ─────────────────────────────────────────────────────────
!python test.py --run_name colab_dwt

In [ ]:
# ── Step 10: View result plots ───────────────────────────────────────────────
from IPython.display import Image, display
import glob

for img_path in sorted(glob.glob('results/plots/colab_dwt/*.png')):
    print(img_path)
    display(Image(img_path))

In [ ]:
# ── Step 11: Save results to Google Drive ────────────────────────────────────
import shutil, os

dst = '/content/drive/MyDrive/WiFi-BullyDetect/results'
os.makedirs(dst, exist_ok=True)
shutil.copytree('results', dst, dirs_exist_ok=True)
print(f'Results saved to Drive: {dst}')